A place to try using the model and understand what's going on (starting at 2026-09-06)

In [10]:
import numpy as np
import pandas as pd
import streamflow_model as S

# 1) load the released (bugged) model and build standardized inputs + target
# train.csv holds the weather/soil inputs AND the observed streamflow; this helper
# builds the 72-hour windowed, z-scored inputs and the aligned target for you
weights = S.load_weights("model/streamflow_model_bug.npz")
feats = S.build_features("data/train.csv", "model/feature_scaler.json")
X, truth = feats["X"], feats["truth"]
print(f"{len(X)} usable examples, {X.shape[1]} features each")

[3.05 2.03 0.76 0.51 0.25 0.  ]
                 datetime  rain_mm  T_air_C  RH_pct  srad_Wm2  Tsoil_4in_C  \
0     2023-10-29 11:00:00      0.0      0.0    85.6       0.0          6.6   
1     2023-10-29 12:00:00      0.0     -0.1    85.3       0.0          6.5   
2     2023-10-29 13:00:00      0.0     -0.3    83.7       2.0          6.3   
3     2023-10-29 14:00:00      0.0     -0.3    82.1      42.6          6.2   
4     2023-10-29 15:00:00      0.0      0.0    80.2      76.8          6.2   
...                   ...      ...      ...     ...       ...          ...   
18680 2025-12-16 14:00:00      0.0     -8.9    84.1      10.2          0.7   
18681 2025-12-16 15:00:00      0.0     -7.5    82.9      85.2          0.7   
18682 2025-12-16 16:00:00      0.0     -5.1    79.4     171.4          0.7   
18683 2025-12-16 17:00:00      0.0     -2.8    75.3     208.2          0.7   
18684 2025-12-16 18:00:00      0.0     -0.3    69.9     330.7          0.7   

       sm_2in  sm_20in    seaso

In [22]:
weights


{'fc1.weight': array([[-7.6256139e-04, -2.3746791e-03, -8.8769691e-03, ...,
          8.5218757e-02, -1.6181801e-02,  7.2891386e-03],
        [-3.1426749e-03,  7.9153047e-04, -3.6873659e-03, ...,
          1.3013140e-03, -5.1770508e-03, -2.3167159e-03],
        [ 2.4503397e-03, -4.7176899e-03, -2.0749883e-03, ...,
          8.7047644e-02, -1.8316310e-02,  9.1621485e-03],
        ...,
        [ 6.6765735e-04,  7.9982396e-04,  4.5840577e-03, ...,
         -1.2756642e-03, -1.2584943e-02, -8.9259371e-03],
        [ 6.2570921e-03,  3.9026842e-03, -6.4627454e-03, ...,
          8.7492988e-02, -2.6787795e-02,  2.8941799e-05],
        [ 9.4705191e-04,  2.8219615e-04,  1.1736418e-03, ...,
          1.3278663e-05, -1.3123732e-03,  1.0234135e-04]],
       shape=(56, 77), dtype=float32),
 'fc1.bias': array([ 9.7290548e-03, -6.1009484e-03,  1.3537866e-02,  3.5892490e-02,
         3.6433190e-02, -1.6346129e-02, -1.9985953e-02,  9.6243730e-06,
        -1.9834816e-02, -1.1506662e-04, -9.7885248e-05, -

In [21]:
for name, arr in acts.items():
    print(name, arr.shape)

fc1 (18685, 56)
fc2 (18685, 56)
fc3 (18685, 56)
fc4 (18685, 56)
fc5 (18685, 56)
fc6 (18685, 56)


In [12]:
feats
# truth: truth streamflow value of each training data


{'X': array([[ 4.2192993 ,  2.7683449 ,  0.96176416, ..., -0.51768625,
         -0.5767691 , -0.65030074],
        [ 2.7683449 ,  0.96176416,  0.606138  , ..., -0.5185916 ,
         -0.58524597, -0.65030074],
        [ 0.96176416,  0.606138  ,  0.23628683, ..., -0.51949656,
         -0.6021998 , -0.64080656],
        ...,
        [-0.11933929, -0.11933929, -0.11933929, ..., -1.2302608 ,
         -1.0090902 ,  0.16334933],
        [-0.11933929, -0.11933929, -0.11933929, ..., -1.2305154 ,
         -0.81412184,  0.338042  ],
        [-0.11933929, -0.11933929, -0.11933929, ..., -1.2307692 ,
         -0.6021998 ,  0.9195598 ]], shape=(18685, 77), dtype=float32),
 'row_id': array([    0,     1,     2, ..., 18682, 18683, 18684], shape=(18685,)),
 'datetime': array(['2023-10-29T11:00:00.000000', '2023-10-29T12:00:00.000000',
        '2023-10-29T13:00:00.000000', ..., '2025-12-16T16:00:00.000000',
        '2025-12-16T17:00:00.000000', '2025-12-16T18:00:00.000000'],
       shape=(18685,), dtype=

In [13]:
train = pd.read_csv("data/train.csv")

In [14]:
train

,datetime,rain_mm,T_air_C,RH_pct,srad_Wm2,Tsoil_4in_C,sm_2in,sm_20in,season,streamflow_mm_hr
0,2023-10-26 12:00:00,3.05,13.1,98.0,42.3,11.6,0.10,0.07,0.287149,0.034445
1,2023-10-26 13:00:00,2.03,13.3,98.0,42.3,11.6,0.10,0.07,0.286825,0.033318
2,2023-10-26 14:00:00,0.76,13.6,98.0,42.3,11.6,0.10,0.07,0.286501,0.031673
3,2023-10-26 15:00:00,0.51,14.4,98.2,42.3,11.6,0.10,0.07,0.286177,0.032089
4,2023-10-26 16:00:00,0.25,15.5,98.2,42.3,11.6,0.10,0.07,0.285853,0.032229
...,...,...,...,...,...,...,...,...,...,...
18751,2025-12-16 14:00:00,0.00,-8.9,84.1,10.2,0.7,0.06,0.06,0.015830,0.000410
18752,2025-12-16 15:00:00,0.00,-7.5,82.9,85.2,0.7,0.06,0.06,0.015741,0.000409
18753,2025-12-16 16:00:00,0.00,-5.1,79.4,171.4,0.7,0.06,0.06,0.015651,0.000407
18754,2025-12-16 17:00:00,0.00,-2.8,75.3,208.2,0.7,0.06,0.06,0.015563,0.000405


In [15]:
#check how does the feature_scalar.json is generated
last_var = train["srad_Wm2"][71:] #returns series

In [16]:
last_var.mean()

np.float64(141.7237195611437)

In [17]:
# 2) run the model, and grab hidden activations (for your analysis)
pred, acts = S.forward(weights, X, return_hidden=True)
print("layers with activations:", list(acts.keys()))

layers with activations: ['fc1', 'fc2', 'fc3', 'fc4', 'fc5', 'fc6']


In [18]:
W, b = weights["fc1.weight"], weights["fc1.bias"]
x = X[0]                              # one hour, shape (77,)
manual = np.maximum(0, W @ x + b)     # (56,77) @ (77,) -> (56,)
print(np.allclose(manual, acts["fc1"][0]))

True


In [23]:
a = acts["fc1"]
print((a == 0).mean())                 # fraction of all entries that are silent
print((a > 0).mean(axis=0).round(2))   # per-neuron: fraction of hours it fires

0.554023471845254
[0.49 0.22 0.49 0.72 0.55 0.22 0.16 0.64 0.18 0.15 0.42 0.15 0.52 0.16
 0.72 0.48 0.18 0.25 0.7  0.16 0.19 0.69 0.52 0.61 0.52 0.51 0.26 0.52
 0.19 0.6  0.51 0.16 0.51 0.7  0.66 0.65 0.17 0.22 0.69 0.16 0.58 0.48
 0.64 0.57 0.64 0.71 0.64 0.4  0.64 0.51 0.58 0.57 0.59 0.2  0.46 0.19]


In [24]:
pred

array([0.03030184, 0.0260518 , 0.0219172 , ..., 0.00782154, 0.00782154,
       0.00782154], shape=(18685,), dtype=float32)

In [25]:
acts

{'fc1': array([[0.        , 0.        , 0.        , ..., 0.        , 0.02310764,
         0.00650362],
        [0.        , 0.        , 0.00367412, ..., 0.        , 0.01878087,
         0.00363632],
        [0.00126431, 0.        , 0.00669809, ..., 0.        , 0.00516277,
         0.00167874],
        ...,
        [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
         0.        ],
        [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
         0.        ],
        [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
         0.        ]], shape=(18685, 56), dtype=float32),
 'fc2': array([[1.75683947e-11, 8.33997888e-07, 4.83879149e-02, ...,
         3.13184228e-06, 5.62207364e-02, 4.84320917e-05],
        [1.45264955e-11, 9.53904191e-07, 3.91136855e-02, ...,
         2.60677825e-06, 4.54209596e-02, 4.95984677e-05],
        [1.18582861e-11, 1.09279313e-06, 3.04938070e-02, ...,
         2.03416607e-06, 3.52958702e-02, 5.07397781e-05],
 

In [26]:
# 3) basic statistics
print("\nprediction stats (mm/hr):")
print(f"mean {pred.mean():.4f} std {pred.std():.4f}"
      f"min {pred.min():.4f} max {pred.max():.4f}")
print(f"skill vs observed: NSE {S.nse(pred, truth):.3f} RMSE {S.rmse(pred, truth):.4f}")


prediction stats (mm/hr):
mean 0.0265 std 0.0264min 0.0078 max 0.5884
skill vs observed: NSE 0.679 RMSE 0.0169


### score how close the predictions are to the observed streamflow

**RMSE, root mean squared error**

`RMSE = sqrt( mean( (pred − truth)² ) )`

Take every hour's error, square it, average, take the square root. It's in the same units as the target, mm/hr, so it answers "how far off is a typical prediction?" Lower is better and 0 is perfect. Squaring means big misses, like getting a flood peak wrong, dominate the score.

here in our case unit: mm/hr

**NSE, Nash–Sutcliffe efficiency**

`NSE = 1 − sum((pred − truth)²) / sum((truth − mean(truth))²)`

This is the hydrology community's standard skill score. The numerator is your model's squared error. The denominator is the squared error you'd get from the dumbest possible model, one that predicts the average streamflow for every hour. So NSE compares you against that baseline:

| NSE | Meaning |
|---|---|
| 1.0 | perfect |
| 0.66 | your errors are 34% of the "predict the mean" errors |
| 0.0 | no better than predicting the mean |
| negative | worse than predicting the mean |

Higher is better. If you know R² from statistics, NSE is the same formula.

In [27]:
# peek at an example weight matrix
print(f"\nfc3.weight: shape {weights['fc3.weight'].shape}")
print(np.round(weights["fc3.weight"][:3, :6], 3)) # change these to see diff values



fc3.weight: shape (56, 56)
[[ 0.    -0.     0.008  0.007  0.008  0.007]
 [-0.    -0.     0.022  0.019  0.023  0.015]
 [ 0.    -0.     0.021  0.017  0.021  0.014]]


In [28]:
# 4) write a (placeholder) submission -> submission.csv  (this is what you upload)
#    feature_direction : (56,)  your recovered leak direction
#    fc_weight_corrected : (56,56) your repaired weight matrix for the affected layer
# (placeholders below score ~0; swap in your recovered direction and repaired matrix)
feature_direction = np.zeros(56, np.float32)             # REPLACE ME
fc_weight_corrected = np.zeros((56, 56), np.float32)     # REPLACE ME
